# Data Mining Lab Assignment 2
## Building a Recommender System using a Medallion Data Lake Architecture

**Student Name:** Aditya Singh Chandel  
**Roll Number:** LCI2023046  
**Course:** B.Tech Data Mining Lab  
**Dataset:** Book-Crossing (Users, Books, Book-Ratings)  
**Execution Environment:** Google Colab / Python 3  

---

### Architecture Overview: The Medallion Lakehouse Paradigm
In modern data engineering and machine learning platforms (such as Databricks and Delta Lake), the **Medallion Architecture** organizes data into three logical tiers to guarantee reliability, auditability, and governance:
1. **Bronze Layer (Raw Ingestion / System of Record):** Append-only, untouched copies of source files enriched with ingestion metadata (ingestion_timestamp). Preserves raw fidelity so changes or bugs can always be reprocessed from ground truth.
2. **Silver Layer (Cleaned, Standardized & Conformed Enterprise Truth):** Deduplicated, validated, and normalized data. Foreign key integrity is strictly enforced, invalid records are imputed or pruned, and interaction sparsity is addressed through activity filtering.
3. **Gold Layer (Curated Feature Store & Model-Ready Matrices):** Aggregated metrics and optimized linear-algebra structures (e.g., Compressed Sparse Row matrices) engineered for machine learning algorithms and real-time inference.

This notebook builds this end-to-end pipeline and deploys an **Item-Based Collaborative Filtering Recommender System** using cosine similarity on high-dimensional sparse user-item interaction spaces.


In [1]:
import os
import sys
import zipfile
import urllib.request
import datetime
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors
from collections import defaultdict

# 1. Initialize Medallion lakehouse directories
for layer in ['bronze', 'silver', 'gold', 'raw_data']:
    os.makedirs(layer, exist_ok=True)
    print(f"Directory ready: {layer}/")

# 2. Download and unpack Book-Crossing dataset if not present
dataset_url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"
zip_target = "book-crossings.zip"

if not os.path.exists(zip_target):
    print("Downloading Book-Crossing zip archive...")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    req = urllib.request.Request(dataset_url, headers=headers)
    with urllib.request.urlopen(req) as resp, open(zip_target, 'wb') as out_f:
        out_f.write(resp.read())
    print("Download complete.")

with zipfile.ZipFile(zip_target, 'r') as zf:
    zf.extractall('raw_data')
    print("Extracted files to raw_data/:", zf.namelist())


Directory ready: bronze/
Directory ready: silver/
Directory ready: gold/
Directory ready: raw_data/
Extracted files to raw_data/: ['BX-Book-Ratings.csv', 'BX-Books.csv', 'BX-Users.csv']


---
## Step 1: Set Up the Data Lake and Build the Bronze Layer

### Bronze Layer Principles:
- Bronze acts as the immutable data lake ingestion tier.
- Data is loaded **exactly as it is** from the raw semicolon-separated, `latin-1` encoded CSV files.
- We append an `ingestion_timestamp` column to maintain an audit trail for data lineage.
- **Zero data cleaning** is conducted in Bronze: all messy values, corrupt strings, and nulls are preserved as-is.
- Tables are stored as columnar **Parquet** files for storage efficiency and rapid downstream querying.


In [2]:
# Ingest raw source files directly into Bronze without cleaning
ingestion_ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
print(f"Ingestion Run Timestamp: {ingestion_ts}")

raw_tables = {
    'users': os.path.join('raw_data', 'BX-Users.csv'),
    'books': os.path.join('raw_data', 'BX-Books.csv'),
    'ratings': os.path.join('raw_data', 'BX-Book-Ratings.csv')
}

bronze_paths = {}
for name, fpath in raw_tables.items():
    print(f"Ingesting {name} from {fpath}...")
    df_raw = pd.read_csv(
        fpath,
        sep=';',
        encoding='latin-1',
        escapechar='\\',
        dtype=str,
        on_bad_lines='skip'
    )
    df_raw['ingestion_timestamp'] = ingestion_ts
    out_parquet = os.path.join('bronze', f"{name}.parquet")
    df_raw.to_parquet(out_parquet, index=False)
    bronze_paths[name] = out_parquet
    print(f" -> Saved {out_parquet} | Shape: {df_raw.shape}")

print("\nBronze Layer Successfully Constructed.")


Ingestion Run Timestamp: 2026-11-23T11:09:15.099042+00:00
Ingesting users from raw_data/BX-Users.csv...
 -> Saved bronze/users.parquet | Shape: (278858, 4)
Ingesting books from raw_data/BX-Books.csv...
 -> Saved bronze/books.parquet | Shape: (271379, 9)
Ingesting ratings from raw_data/BX-Book-Ratings.csv...
 -> Saved bronze/ratings.parquet | Shape: (1149780, 4)

Bronze Layer Successfully Constructed.


---
## Step 2: Explore the Bronze Data (EDA)

We load the Bronze Parquet tables back into memory and execute exploratory data analysis across the 5 designated operational dimensions.
For each item, a clear markdown cell specifies:
1. **What was found in Bronze** (quantitative facts and distributions).
2. **What remediation will be applied in Silver** (explicit cleaning strategy).


### 2.1 Shape, Data Types, and Missing-Value Count


In [3]:
# Read Bronze Parquet tables
bronze_users = pd.read_parquet('bronze/users.parquet')
bronze_books = pd.read_parquet('bronze/books.parquet')
bronze_ratings = pd.read_parquet('bronze/ratings.parquet')

tables = {'Users': bronze_users, 'Books': bronze_books, 'Ratings': bronze_ratings}

for name, df in tables.items():
    print(f"\n{'='*25} {name} Table {'='*25}")
    print(f"Shape: {df.shape}")
    summary = pd.DataFrame({
        'Dtype': df.dtypes,
        'Null_Count': df.isnull().sum(),
        'Null_Pct': (df.isnull().mean() * 100).round(2)
    })
    print(summary)


========================= Users Table =========================
Shape: (278858, 4)
                        Dtype  Null_Count  Null_Pct
User-ID                object           0      0.00
Location               object           0      0.00
Age                    object      110762     39.72
ingestion_timestamp    object           0      0.00

========================= Books Table =========================
Shape: (271379, 9)
                        Dtype  Null_Count  Null_Pct
ISBN                   object           0      0.00
Book-Title             object           0      0.00
Book-Author            object           2      0.00
Year-Of-Publication    object           0      0.00
Publisher              object           2      0.00
Image-URL-S            object           0      0.00
Image-URL-M            object           0      0.00
Image-URL-L            object           0      0.00
ingestion_timestamp    object           0      0.00

========================= Ratings Table ============

> **EDA Finding & Silver Plan:** The raw tables comprise 278,858 users (with 110,762 missing ages, or 39.72%), 271,379 books (with unneeded image URLs and minor author/publisher nulls), and 1,149,780 ratings (zero null values); in Silver, we will parse types to appropriate numeric integers, impute missing user ages, drop all image URL columns, and enforce foreign-key referential integrity.


### 2.2 Distribution of Ratings (Implicit vs. Explicit)


In [4]:
ratings_num = pd.to_numeric(bronze_ratings['Book-Rating'], errors='coerce')
rating_distribution = ratings_num.value_counts().sort_index()

total_ratings = len(ratings_num)
zero_count = rating_distribution.get(0, 0)
explicit_ratings = ratings_num[ratings_num > 0]

print("Rating Distribution (0 = Implicit Interaction, 1-10 = Explicit Score):")
for r, count in rating_distribution.items():
    pct = (count / total_ratings) * 100
    tag = " (Implicit)" if r == 0 else " (Explicit)"
    print(f"  Rating {r:2.0f}: {count:8,d} ({pct:5.2f}%){tag}")

print(f"\nTotal Ratings: {total_ratings:,}")
print(f"Zero Ratings : {zero_count:,} ({(zero_count/total_ratings)*100:.2f}%)")
print(f"Explicit Ratings (1-10): {len(explicit_ratings):,} ({(len(explicit_ratings)/total_ratings)*100:.2f}%)")
print(f"Explicit Summary: Mean = {explicit_ratings.mean():.2f}, Median = {explicit_ratings.median():.1f}, Std = {explicit_ratings.std():.2f}")


Rating Distribution (0 = Implicit Interaction, 1-10 = Explicit Score):
  Rating  0:  716,109 (62.28%) (Implicit)
  Rating  1:    1,770 ( 0.15%) (Explicit)
  Rating  2:    2,759 ( 0.24%) (Explicit)
  Rating  3:    5,790 ( 0.50%) (Explicit)
  Rating  4:    8,904 ( 0.77%) (Explicit)
  Rating  5:   50,674 ( 4.41%) (Explicit)
  Rating  6:   36,924 ( 3.21%) (Explicit)
  Rating  7:   76,457 ( 6.65%) (Explicit)
  Rating  8:  103,736 ( 9.02%) (Explicit)
  Rating  9:   67,541 ( 5.87%) (Explicit)
  Rating 10:   78,866 ( 6.86%) (Explicit)

Total Ratings: 1,149,780
Zero Ratings : 716,109 (62.28%)
Explicit Ratings (1-10): 433,671 (37.72%)
Explicit Summary: Mean = 7.60, Median = 8.0, Std = 1.84


> **EDA Finding & Silver Plan:** Out of 1,149,780 total interactions, 716,109 ratings (62.28%) have a value of 0 denoting implicit reads/clicks rather than explicit evaluations, while explicit scores span 1–10 with a mean of 7.60 and median of 8.0; in Silver, we will filter out all 0 ratings and retain strictly explicit scores (1–10) so collaborative filtering measures genuine user affinity.


### 2.3 Distribution of User Ages and Publication Years


In [5]:
# 1. User Age Inspection
user_ages = pd.to_numeric(bronze_users['Age'], errors='coerce')
age_under_5 = (user_ages < 5).sum()
age_over_100 = (user_ages > 100).sum()
age_nulls = user_ages.isnull().sum()
valid_ages = user_ages[(user_ages >= 5) & (user_ages <= 100)]
median_valid_age = valid_ages.median()

print("User Age Distribution & Anomaly Analysis:")
print(f"  Missing / NaN Ages : {age_nulls:,} ({(age_nulls / len(bronze_users))*100:.2f}%)")
print(f"  Age < 5 (Physiologically impossible)   : {age_under_5:,}")
print(f"  Age > 100 (Outliers / Corrupted entries): {age_over_100:,}")
print(f"  Valid Age Range (5 to 100) Median      : {median_valid_age:.1f}")
print(f"  Valid Age Range Mean                   : {valid_ages.mean():.2f}")

# 2. Book Publication Year Inspection
book_years = pd.to_numeric(bronze_books['Year-Of-Publication'], errors='coerce')
year_zero = (book_years == 0).sum()
year_future = (book_years > 2026).sum()
year_ancient = (book_years < 1800).sum()
year_nulls = book_years.isnull().sum()

print("\nBook Publication Year Anomaly Analysis:")
print(f"  Year = 0 (Unspecified / Placeholder)    : {year_zero:,}")
print(f"  Year > 2026 (Future impossible dates)  : {year_future:,}")
print(f"  Year < 1800 (Historical / OCR glitches): {year_ancient:,}")
print(f"  Non-numeric / NaN Years                : {year_nulls:,}")


User Age Distribution & Anomaly Analysis:
  Missing / NaN Ages : 110,762 (39.72%)
  Age < 5 (Physiologically impossible)   : 882
  Age > 100 (Outliers / Corrupted entries): 366
  Valid Age Range (5 to 100) Median      : 32.0
  Valid Age Range Mean                   : 34.75

Book Publication Year Anomaly Analysis:
  Year = 0 (Unspecified / Placeholder)    : 4,619
  Year > 2026 (Future impossible dates)  : 11
  Year < 1800 (Historical / OCR glitches): 4,621
  Non-numeric / NaN Years                : 0


> **EDA Finding & Silver Plan:** User ages contain biological anomalies including 882 users under age 5 and 366 users over age 100 (alongside 39.72% missing ages with valid median 32), while publication years contain 4,619 zeros and 11 future dates beyond 2026; in Silver, we will replace user ages below 5 or above 100 with NaN and impute them with the median (32), while setting invalid publication years (<= 0 or > 2026) to null.


### 2.4 Number of Ratings per User and per Book


In [6]:
explicit_df = bronze_ratings[ratings_num > 0].copy()
user_counts = explicit_df['User-ID'].value_counts()
book_counts = explicit_df['ISBN'].str.strip().str.upper().value_counts()

print("User Rating Activity (Explicit Ratings) Quantiles:")
print(user_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))
print(f"Users with >= 10 ratings: {(user_counts >= 10).sum():,} out of {len(user_counts):,} ({(user_counts >= 10).sum()/len(user_counts)*100:.2f}%)")

print("\nBook Rating Popularity (Explicit Ratings) Quantiles:")
print(book_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))
print(f"Books with >= 10 ratings: {(book_counts >= 10).sum():,} out of {len(book_counts):,} ({(book_counts >= 10).sum()/len(book_counts)*100:.2f}%)")


User Rating Activity (Explicit Ratings) Quantiles:
0.25     1.0
0.50     1.0
0.75     3.0
0.90     9.0
0.95    19.0
0.99    73.0
Name: count, dtype: float64
Users with >= 10 ratings: 7,334 out of 77,805 (9.43%)

Book Rating Popularity (Explicit Ratings) Quantiles:
0.25     1.0
0.50     1.0
0.75     2.0
0.90     4.0
0.95     6.0
0.99    22.0
Name: count, dtype: float64
Books with >= 10 ratings: 5,647 out of 185,841 (3.04%)


> **EDA Finding & Silver Plan:** Interaction volume follows an extreme long-tail power law distribution where over 50% of users and books have only a single rating, and merely 9.43% of users and 3.04% of books possess 10 or more ratings; in Silver, we will prune users and books with fewer than 10 ratings to eliminate cold-start noise and ensure vectors have sufficient common support for cosine similarity.


### 2.5 Matrix Sparsity


In [7]:
n_users_bronze = bronze_ratings['User-ID'].nunique()
n_books_bronze = bronze_ratings['ISBN'].nunique()
n_ratings_bronze = len(bronze_ratings)

total_possible_cells = n_users_bronze * n_books_bronze
matrix_density = (n_ratings_bronze / total_possible_cells) * 100
matrix_sparsity = 100.0 - matrix_density

print("Raw Rating Matrix Dimension & Sparsity Calculation:")
print(f"  Unique Users in Ratings  (Rows/Cols) : {n_users_bronze:,}")
print(f"  Unique Books in Ratings  (Cols/Rows) : {n_books_bronze:,}")
print(f"  Total Possible Interaction Pairs     : {total_possible_cells:,}")
print(f"  Observed Interaction Pairs           : {n_ratings_bronze:,}")
print(f"  Interaction Matrix Density           : {matrix_density:.6f}%")
print(f"  Interaction Matrix Sparsity          : {matrix_sparsity:.6f}%")


Raw Rating Matrix Dimension & Sparsity Calculation:
  Unique Users in Ratings  (Rows/Cols) : 105,283
  Unique Books in Ratings  (Cols/Rows) : 340,556
  Total Possible Interaction Pairs     : 35,854,757,348
  Observed Interaction Pairs           : 1,149,780
  Interaction Matrix Density           : 0.003207%
  Interaction Matrix Sparsity          : 99.996793%


> **EDA Finding & Silver Plan:** The raw interaction matrix is 99.9968% sparse with only 0.0032% of 35.85 billion potential user-book coordinates populated; in Silver, filtering out low-activity entities will elevate density to ~0.25% (an 80-fold density enhancement), turning an intractable, uninformative space into a dense, model-ready interaction manifold.


---
## Step 3: Clean the Data and Build the Silver Layer

### Transformation Pipeline & Business Rules:
1. **Deduplication:** Remove duplicate records across users, books, and user-book rating interactions.
2. **Explicit Interaction Filtering:** Discard all `0` ratings; retain strictly explicit evaluations from 1 to 10.
3. **Age Validation & Median Imputation:** Replace user ages $< 5$ or $> 100$ with `NaN`, then impute all missing values with the median valid age ($32$).
4. **Location Parsing:** Dissect the semicolon/comma `Location` field into three conformed dimensions: `city`, `state`, and `country`, stripping whitespace and title-casing.
5. **Publication Year Sanity:** Set publication years $\le 0$ or $> 2026$ (future impossible dates) to `NaN`.
6. **Text Normalization & Schema Pruning:** Strip superfluous whitespace and casing anomalies from book titles, authors, and publishers. Drop unneeded image URLs (`Image-URL-S`, `Image-URL-M`, `Image-URL-L`).
7. **Foreign Key Integrity:** Enforce strict referential constraints: ratings must reference an existing book in the books catalog and an existing user in the users table.
8. **Minimum Interaction Threshold Justification ($k = 10$):**
   - In Step 2.4, we proved that $90.57\%$ of users have $< 10$ ratings and $96.96\%$ of books have $< 10$ ratings.
   - If an item has only 1 or 2 ratings, computing cosine similarity against other items results in degenerate dot products (often zero or spuriously high due to coincidental overlap).
   - Establishing $k = 10$ retains substantial volume while boosting interaction density by $80\times$, guaranteeing statistically dependable neighborhoods for collaborative filtering.


In [8]:
# Load raw Bronze Parquet tables
u_bronze = pd.read_parquet('bronze/users.parquet')
b_bronze = pd.read_parquet('bronze/books.parquet')
r_bronze = pd.read_parquet('bronze/ratings.parquet')

# -------------------------------------------------------------------------
# 1. Clean Users
# -------------------------------------------------------------------------
users_clean = u_bronze.drop(columns=['ingestion_timestamp'], errors='ignore').copy()
users_clean['User-ID'] = pd.to_numeric(users_clean['User-ID'], errors='coerce')
users_clean = users_clean.dropna(subset=['User-ID']).copy()
users_clean['User-ID'] = users_clean['User-ID'].astype(np.int64)
users_clean = users_clean.drop_duplicates(subset=['User-ID']).copy()

# Fix Ages: set < 5 or > 100 to NaN, impute with median of valid ages
users_clean['Age'] = pd.to_numeric(users_clean['Age'], errors='coerce')
users_clean.loc[(users_clean['Age'] < 5) | (users_clean['Age'] > 100), 'Age'] = np.nan
median_age_silver = users_clean.loc[users_clean['Age'].notna(), 'Age'].median()
users_clean['Age'] = users_clean['Age'].fillna(median_age_silver).astype(int)

# Split Location into city, state, country
loc_split = users_clean['Location'].astype(str).str.split(',', n=2, expand=True)
users_clean['city'] = loc_split[0].str.strip().str.title().replace(['', 'N/A', 'Nan'], np.nan)
users_clean['state'] = (loc_split[1].str.strip().str.title().replace(['', 'N/A', 'Nan'], np.nan) if 1 in loc_split.columns else np.nan)
users_clean['country'] = (loc_split[2].str.strip().str.title().replace(['', 'N/A', 'Nan'], np.nan) if 2 in loc_split.columns else np.nan)

# -------------------------------------------------------------------------
# 2. Clean Books
# -------------------------------------------------------------------------
books_clean = b_bronze.drop(columns=['ingestion_timestamp'], errors='ignore').copy()
books_clean['ISBN'] = books_clean['ISBN'].astype(str).str.strip().str.upper()
books_clean = books_clean.drop_duplicates(subset=['ISBN']).copy()

# Fix Publication Years: set <= 0 or > 2026 to NaN
pub_years = pd.to_numeric(books_clean['Year-Of-Publication'], errors='coerce')
books_clean['Year-Of-Publication'] = pub_years.apply(lambda y: int(y) if (pd.notna(y) and 1800 <= y <= 2026) else np.nan)

# Clean text columns and drop unneeded image URLs
books_clean.drop(columns=['Image-URL-S', 'Image-URL-M', 'Image-URL-L'], inplace=True, errors='ignore')
books_clean['Book-Title'] = books_clean['Book-Title'].astype(str).str.strip()
books_clean['Book-Author'] = books_clean['Book-Author'].astype(str).str.strip()
books_clean['Publisher'] = books_clean['Publisher'].astype(str).str.strip()

# -------------------------------------------------------------------------
# 3. Clean Ratings
# -------------------------------------------------------------------------
ratings_clean = r_bronze.drop(columns=['ingestion_timestamp'], errors='ignore').copy()
ratings_clean['User-ID'] = pd.to_numeric(ratings_clean['User-ID'], errors='coerce')
ratings_clean['ISBN'] = ratings_clean['ISBN'].astype(str).str.strip().str.upper()
ratings_clean['Book-Rating'] = pd.to_numeric(ratings_clean['Book-Rating'], errors='coerce')
ratings_clean = ratings_clean.dropna(subset=['User-ID', 'ISBN', 'Book-Rating']).copy()
ratings_clean['User-ID'] = ratings_clean['User-ID'].astype(np.int64)
ratings_clean['Book-Rating'] = ratings_clean['Book-Rating'].astype(int)

# Deduplicate interactions
ratings_clean = ratings_clean.drop_duplicates(subset=['User-ID', 'ISBN']).copy()

# Discard 0 ratings, retain explicit ratings between 1 and 10
ratings_clean = ratings_clean[(ratings_clean['Book-Rating'] >= 1) & (ratings_clean['Book-Rating'] <= 10)].copy()

# Referential integrity constraint
ratings_clean = ratings_clean[
    ratings_clean['User-ID'].isin(users_clean['User-ID']) & 
    ratings_clean['ISBN'].isin(books_clean['ISBN'])
].copy()

# -------------------------------------------------------------------------
# 4. Filter Minimum Rating Threshold (k >= 10)
# -------------------------------------------------------------------------
min_threshold = 10
user_activity = ratings_clean['User-ID'].value_counts()
book_popularity = ratings_clean['ISBN'].value_counts()

active_users = user_activity[user_activity >= min_threshold].index
active_books = book_popularity[book_popularity >= min_threshold].index

ratings_silver = ratings_clean[
    ratings_clean['User-ID'].isin(active_users) & 
    ratings_clean['ISBN'].isin(active_books)
].copy()

# Align users and books catalog with silver ratings
users_silver = users_clean[users_clean['User-ID'].isin(ratings_silver['User-ID'].unique())].copy()
books_silver = books_clean[books_clean['ISBN'].isin(ratings_silver['ISBN'].unique())].copy()

# Save Cleaned Silver Tables to Parquet
users_silver.to_parquet('silver/users_silver.parquet', index=False)
users_silver.to_parquet('silver/users.parquet', index=False)
books_silver.to_parquet('silver/books_silver.parquet', index=False)
books_silver.to_parquet('silver/books.parquet', index=False)
ratings_silver.to_parquet('silver/ratings_silver.parquet', index=False)
ratings_silver.to_parquet('silver/ratings.parquet', index=False)

print(f"=== SILVER LAYER TRANSFORMATION SUMMARY ===")
print(f"Users Silver   : {users_silver.shape} (Age nulls: {users_silver['Age'].isnull().sum()})")
print(f"Books Silver   : {books_silver.shape} (Columns: {list(books_silver.columns)})")
print(f"Ratings Silver : {ratings_silver.shape} (Rating Range: {ratings_silver['Book-Rating'].min()} to {ratings_silver['Book-Rating'].max()})")

density_silver = (len(ratings_silver) / (len(users_silver) * len(books_silver))) * 100
print(f"Silver Matrix Density : {density_silver:.4f}% (Sparsity: {100 - density_silver:.4f}%)")


=== SILVER LAYER TRANSFORMATION SUMMARY ===
Users Silver   : (6319, 6) (Age nulls: 0)
Books Silver   : (5440, 5) (Columns: ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher'])
Ratings Silver : (87233, 3) (Rating Range: 1 to 10)
Silver Matrix Density : 0.2538% (Sparsity: 99.7462%)


---
## Step 4: Build the Gold Layer

### Gold Layer Model-Ready Assets:
In the Gold layer, we construct consumption-optimized analytical assets:
1. **Book Lookup Catalog (`gold/book_lookup.parquet`):** Clean metadata mapping each unique `ISBN` to its readable `Book-Title`, `Book-Author`, `Year-Of-Publication`, and `Publisher`.
2. **Entity-Level Aggregates (`gold/user_aggregates.parquet` & `gold/book_aggregates.parquet`):** Summary metrics (rating count, average rating, min, and max) computed per user and per book.
3. **User-Book Sparse Rating Matrix (`gold/book_user_matrix.npz`):** A Scipy Compressed Sparse Row (`csr_matrix`) holding explicit ratings where rows correspond to books and columns correspond to users, allowing instantaneous cosine similarity matrix vector operations.


In [9]:
# Load cleaned Silver tables
users_s = pd.read_parquet('silver/users_silver.parquet')
books_s = pd.read_parquet('silver/books_silver.parquet')
ratings_s = pd.read_parquet('silver/ratings_silver.parquet')

# 1. Book Lookup Table
book_lookup = books_s[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher']].copy()
book_lookup.to_parquet('gold/book_lookup.parquet', index=False)

# 2. User & Book Aggregates
user_aggs = ratings_s.groupby('User-ID')['Book-Rating'].agg(
    rating_count='count',
    avg_rating='mean',
    min_rating='min',
    max_rating='max'
).reset_index()
user_aggs.to_parquet('gold/user_aggregates.parquet', index=False)

book_aggs = ratings_s.groupby('ISBN')['Book-Rating'].agg(
    rating_count='count',
    avg_rating='mean',
    min_rating='min',
    max_rating='max'
).reset_index()
book_aggs.to_parquet('gold/book_aggregates.parquet', index=False)

# 3. Build User-Book / Book-User Sparse Rating Matrix
unique_users = np.sort(ratings_s['User-ID'].unique())
unique_books = np.sort(ratings_s['ISBN'].unique())

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
book_to_idx = {isbn: i for i, isbn in enumerate(unique_books)}
idx_to_user = {i: uid for uid, i in user_to_idx.items()}
idx_to_book = {i: isbn for isbn, i in book_to_idx.items()}

# Persist mappings
pd.DataFrame({'user_id': unique_users, 'user_idx': np.arange(len(unique_users))}).to_parquet('gold/user_index_mapping.parquet', index=False)
pd.DataFrame({'isbn': unique_books, 'book_idx': np.arange(len(unique_books))}).to_parquet('gold/book_index_mapping.parquet', index=False)

row_indices = ratings_s['ISBN'].map(book_to_idx).values
col_indices = ratings_s['User-ID'].map(user_to_idx).values
ratings_values = ratings_s['Book-Rating'].values

# Book-by-User sparse CSR matrix: rows = books (5,440), columns = users (6,319)
book_user_matrix = csr_matrix(
    (ratings_values, (row_indices, col_indices)),
    shape=(len(unique_books), len(unique_users)),
    dtype=np.float32
)

save_npz('gold/book_user_matrix.npz', book_user_matrix)
save_npz('gold/user_book_matrix.npz', book_user_matrix.T.tocsr())

print(f"=== GOLD LAYER ASSETS GENERATED ===")
print(f"Book Lookup Shape        : {book_lookup.shape}")
print(f"User Aggregates Shape    : {user_aggs.shape} | Avg ratings/user: {user_aggs['rating_count'].mean():.1f}")
print(f"Book Aggregates Shape    : {book_aggs.shape} | Avg ratings/book: {book_aggs['rating_count'].mean():.1f}")
print(f"Book-User Matrix Shape   : {book_user_matrix.shape} (Rows: Books, Cols: Users)")
print(f"Non-Zero Elements (NNZ)  : {book_user_matrix.nnz:,}")


=== GOLD LAYER ASSETS GENERATED ===
Book Lookup Shape        : (5440, 5)
User Aggregates Shape    : (6319, 5) | Avg ratings/user: 13.8
Book Aggregates Shape    : (5440, 5) | Avg ratings/book: 16.0
Book-User Matrix Shape   : (5440, 6319) (Rows: Books, Cols: Users)
Non-Zero Elements (NNZ)  : 87,233


---
## Step 5: Build the Recommender (Item-Based Collaborative Filtering)

### Mathematical Formulation & Algorithmic Design:
In **Item-Based Collaborative Filtering**, each book $i$ is modeled as a sparse rating vector $\vec{r}_i \in \mathbb{R}^M$ across all $M$ users.
The affinity between book $i$ and candidate book $j$ is quantified using **Cosine Similarity**:

$$\text{Cosine Similarity}(i, j) = \cos(\vec{r}_i, \vec{r}_j) = \frac{\vec{r}_i \cdot \vec{r}_j}{\|\vec{r}_i\|_2 \|\vec{r}_j\|_2} = \frac{\sum_{u=1}^M r_{u, i} \cdot r_{u, j}}{\sqrt{\sum_{u=1}^M r_{u, i}^2} \sqrt{\sum_{u=1}^M r_{u, j}^2}}$$

### Recommendation Scoring:
For a target user $u$, we retrieve the subset of books they rated highly, $\mathcal{H}_u = \{i \mid r_{u, i} \ge \tau\}$. For each candidate book $j \notin \text{Read}(u)$, we calculate an aggregated preference score weighted by the user's historical rating:

$$\text{Score}(u, j) = \sum_{i \in \mathcal{H}_u} \text{sim}(i, j) \cdot r_{u, i}$$

Candidate items are ranked in descending order of $\text{Score}(u, j)$, and the top-$n$ recommendations are presented with clean book metadata.


In [10]:
# 1. Load Gold Matrix and Models
book_user_mat = load_npz('gold/book_user_matrix.npz')
book_lookup_df = pd.read_parquet('gold/book_lookup.parquet')
book_meta = book_lookup_df.set_index('ISBN').to_dict('index')

# 2. Fit NearestNeighbors with Cosine Metric on Sparse Matrix
nn_model = NearestNeighbors(metric='cosine', algorithm='brute')
nn_model.fit(book_user_mat)
print("NearestNeighbors Cosine Model successfully fitted on Gold sparse matrix.")

# 3. Function: get_similar_books(book_title, n=10)
def get_similar_books(book_title: str, n: int = 10):
    '''Returns the top n most similar books to a target book title using cosine similarity.'''
    query = book_title.strip().lower()
    matches = [isbn for isbn, meta in book_meta.items() if query in meta['Book-Title'].lower()]
    if not matches:
        return f"Book title matching '{book_title}' not found in catalog."
    
    target_isbn = matches[0]
    target_idx = book_to_idx[target_isbn]
    
    # Query Nearest Neighbors (n+1 to account for self-match)
    distances, indices = nn_model.kneighbors(book_user_mat[target_idx], n_neighbors=n+1)
    
    similar_books = []
    for dist, idx in zip(distances[0][1:], indices[0][1:]):
        cand_isbn = idx_to_book[idx]
        info = book_meta[cand_isbn]
        similar_books.append({
            'ISBN': cand_isbn,
            'Title': info['Book-Title'],
            'Author': info['Book-Author'],
            'Year': info['Year-Of-Publication'],
            'Cosine_Similarity': round(float(1.0 - dist), 4)
        })
    return pd.DataFrame(similar_books)

# 4. Function: recommend(user_id, n=10, min_user_rating=7, sim_threshold=0.05)
def recommend(user_id: int, n: int = 10, min_user_rating: int = 7, sim_threshold: float = 0.05):
    '''Recommends top n books for a user based on rating-weighted item similarities.'''
    u_ratings = ratings_silver[ratings_silver['User-ID'] == user_id]
    if u_ratings.empty:
        return f"User {user_id} not found in model dataset."
    
    read_isbns = set(u_ratings['ISBN'].tolist())
    
    # Select user's highly rated books (rating >= 7)
    high_rated = u_ratings[u_ratings['Book-Rating'] >= min_user_rating]
    if len(high_rated) < 3:
        high_rated = u_ratings.sort_values(by='Book-Rating', ascending=False).head(5)
        
    candidate_scores = defaultdict(float)
    candidate_max_sim = defaultdict(float)
    
    for _, row in high_rated.iterrows():
        b_isbn = row['ISBN']
        b_rating = row['Book-Rating']
        if b_isbn not in book_to_idx:
            continue
        idx = book_to_idx[b_isbn]
        
        # Retrieve nearest 15 neighbor items
        distances, indices = nn_model.kneighbors(book_user_mat[idx], n_neighbors=16)
        for dist, n_idx in zip(distances[0][1:], indices[0][1:]):
            n_isbn = idx_to_book[n_idx]
            if n_isbn in read_isbns:
                continue
            sim = 1.0 - dist
            if sim >= sim_threshold:
                candidate_scores[n_isbn] += sim * b_rating
                if sim > candidate_max_sim[n_isbn]:
                    candidate_max_sim[n_isbn] = sim
                    
    # Fallback to top neighbors if threshold excluded all candidates
    if not candidate_scores:
        for _, row in high_rated.head(5).iterrows():
            b_isbn = row['ISBN']
            b_rating = row['Book-Rating']
            if b_isbn not in book_to_idx:
                continue
            idx = book_to_idx[b_isbn]
            distances, indices = nn_model.kneighbors(book_user_mat[idx], n_neighbors=8)
            for dist, n_idx in zip(distances[0][1:], indices[0][1:]):
                n_isbn = idx_to_book[n_idx]
                if n_isbn not in read_isbns:
                    candidate_scores[n_isbn] += (1.0 - dist) * b_rating
                    candidate_max_sim[n_isbn] = max(candidate_max_sim[n_isbn], 1.0 - dist)
                    
    ranked = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:n]
    
    recommendations = []
    for rank, (cand_isbn, score) in enumerate(ranked, 1):
        info = book_meta.get(cand_isbn, {})
        recommendations.append({
            'Rank': rank,
            'ISBN': cand_isbn,
            'Title': info.get('Book-Title', 'Unknown'),
            'Author': info.get('Book-Author', 'Unknown'),
            'Year': info.get('Year-Of-Publication', 'Unknown'),
            'Weighted_Score': round(score, 3),
            'Max_Cosine_Sim': round(candidate_max_sim[cand_isbn], 4)
        })
    return pd.DataFrame(recommendations)

# Sanity test: get_similar_books
print("=== Top 5 Similar Books to 'The Fellowship of the Ring' ===")
fellowship_recs = get_similar_books("The Fellowship of the Ring", n=5)
print(fellowship_recs[['Title', 'Author', 'Cosine_Similarity']].to_string(index=False))


NearestNeighbors Cosine Model successfully fitted on Gold sparse matrix.
=== Top 5 Similar Books to 'The Fellowship of the Ring' ===
                                                     Title         Author  Cosine_Similarity
    The Return of the King (The Lord of the Rings, Part 3) J.R.R. TOLKIEN             0.4756
            The Two Towers (The Lord of the Rings, Part 2) J.R.R. TOLKIEN             0.4444
The Hobbit : The Enchanting Prelude to The Lord of the Rings J.R.R. TOLKIEN             0.2904
                                          The Silmarillion J.R.R. TOLKIEN             0.1202
                                          Crazy in Alabama Mark Childress             0.1183


---
## Step 6: Demo and Sanity Check

To evaluate the real-world recommendation quality and personalization of the model, we run `recommend()` across **three distinct users** with diverse literary preferences:
1. **User 139913 (Mystery & Crime Thriller Aficionado):** Strong affinity for detective mysteries, including Sue Grafton's Kinsey Millhone series.
2. **User 78553 (Classic Literature & Fantasy Fan):** Reader of timeless classics (*The Secret Garden*, *The Hobbit*, *Harry Potter*).
3. **User 56959 (Literary & Contemporary Fiction Reader):** Enjoys acclaimed contemporary novels (*One Hundred Years of Solitude*, *Hunting Season*).

For each user, we print their **top highly rated books** alongside the **recommender engine's top 10 suggestions**.


In [11]:
demo_users = [139913, 78553, 56959]

for uid in demo_users:
    print("\n" + "=" * 90)
    print(f" DEMONSTRATION & PERSONALIZATION SANITY CHECK FOR USER ID: {uid}")
    print("=" * 90)
    
    # 1. Print books the user rated highly
    user_top = ratings_silver[ratings_silver['User-ID'] == uid].sort_values(by='Book-Rating', ascending=False).head(5)
    print("\n[Books Rated Highly by User]:")
    for _, row in user_top.iterrows():
        meta = book_meta.get(row['ISBN'], {})
        print(f"  * Rating {row['Book-Rating']:2d}/10 | '{meta.get('Book-Title', 'Unknown')}' by {meta.get('Book-Author', 'Unknown')} (ISBN: {row['ISBN']})")
        
    # 2. Generate and print recommendations
    recs = recommend(uid, n=10)
    print("\n[Top 10 Personalized Book Recommendations]:")
    print(recs[['Rank', 'Title', 'Author', 'Weighted_Score', 'Max_Cosine_Sim']].to_string(index=False))


 DEMONSTRATION & PERSONALIZATION SANITY CHECK FOR USER ID: 139913

[Books Rated Highly by User]:
  * Rating 10/10 | 'Beach House' by Mary Alice Monroe (ISBN: 1551668998)
  * Rating 10/10 | 'The Third Victim' by LISA GARDNER (ISBN: 0553578685)
  * Rating 10/10 | 'Your Oasis on Flame Lake (Ballantine Reader's Circle)' by LORNA LANDVIK (ISBN: 0449002985)
  * Rating 10/10 | 'The Notebook' by Nicholas Sparks (ISBN: 0446605239)
  * Rating 10/10 | 'L Is for Lawless' by Sue Grafton (ISBN: 0449221490)

[Top 10 Personalized Book Recommendations]:
 Rank                                                                                Title             Author  Weighted_Score  Max_Cosine_Sim
    1 E Is for Evidence: A Kinsey Millhone Mystery (Kinsey Millhone Mysteries (Paperback))        Sue Grafton          28.411          0.5279
    2                                                                    I Is for Innocent        Sue Grafton          28.393          0.4706
    3                          

### Qualitative Sanity Check & Analytical Discussion

The empirical outputs above provide compelling verification of our Item-Based Collaborative Filtering engine:

1. **User 139913 (Crime & Mystery Preference):**
   - **Input Profile:** The user assigned a perfect $10/10$ to *L Is for Lawless* by Sue Grafton, as well as thrillers by Lisa Gardner and Nicholas Sparks.
   - **Recommender Output:** The system identified *E Is for Evidence*, *I Is for Innocent*, *K Is for Killer*, and *A Is for Alibi*—all by Sue Grafton—occupying the top 4 recommendation slots with significant cosine similarities ($0.31$ to $0.53$) and weighted scores ($16.1$ to $28.4$).
   - **Sanity Conclusion:** Highly successful. The model accurately clustered the Kinsey Millhone alphabet detective series based strictly on community rating vectors without any explicit textual knowledge of the author or genre.

2. **User 78553 (Classic / Fantasy Preference):**
   - **Input Profile:** Highly rated classic young-adult and fantasy literature (*The Secret Garden*, *The Hobbit*, *Harry Potter*).
   - **Recommender Output:** The engine surfaced classic romance and adventure fiction by Johanna Lindsey (*Glorious Angel*, *Secret Fire*, *Prisoner of My Desire*) and Anne McCaffrey (*The Chronicles of Pern*).
   - **Sanity Conclusion:** The system leveraged shared readership patterns among avid fantasy and historical fiction readers, presenting books with high neighborhood overlap.

3. **User 56959 (Eclectic Literary & Contemporary Reader):**
   - **Input Profile:** A varied reading history spanning magical realism (*One Hundred Years of Solitude*), political satire (*Al Franken*), and mystery fiction (*Nevada Barr*).
   - **Recommender Output:** Produced an intriguing blend of European literary fiction (*Alessandro Baricco*, *Wilhelm Genazino*), comic literature (*Bill Watterson*), and investigative mysteries (*J.A. Jance*, *Jan Burke*).
   - **Sanity Conclusion:** The rating-weighted aggregation gracefully blended the user's multi-faceted interests, promoting crime mysteries alongside reflective literature.

### Key Architectural Takeaways:
- **Medallion Data Lake Integrity:** The Bronze layer ensured total reproducibility, the Silver layer cured severe data defects (null ages, impossible publication years, unstandardized locations, implicit 0-ratings, and extreme 99.997% sparsity), and the Gold layer provided an ultra-fast, memory-efficient sparse CSR matrix for model training.
- **Collaborative Filtering Efficacy:** Item-item collaborative filtering proved remarkably resilient, demonstrating high semantic coherence and avoiding cold-start noise through the $k=10$ pruning threshold.
